# Meta-Learning Pipeline v2

**Combines the best of `importopenml.ipynb` + `DataCollector2.py`**

### Key Improvements over v1
1. **Leakage-Free Evaluation**: All preprocessing is applied WITHIN each CV fold (fit on train, transform val)
2. **Constrained Proxy Model**: `max_depth=5` + early stopping, forces model to rely on feature engineering
3. **Smoothed Target Encoding**: Bayesian smoothing prevents overfitting on rare categories
4. **Unseen Category Handling**: Validation set categories not seen in training are handled gracefully
5. **Rich Meta-Features**: 70 dataset-level meta-features for the meta-model

### Pipeline Overview
1. **Step 1**: Define leakage-free evaluation engine + preprocessing methods
2. **Step 2**: Define meta-feature extraction (70 features)
3. **Step 3**: Collect performance labels from OpenML CC18 (Suite 99)
4. **Step 4**: Train meta-model regressors (one per preprocessing method)
5. **Step 5**: Inference — recommend preprocessing for a new dataset

### Data Source
- **OpenML-CC18** (Suite 99): 72 curated classification benchmark datasets

In [1]:
import openml
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import early_stopping
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import (
    StandardScaler, PowerTransformer, LabelEncoder,
    OneHotEncoder as SklearnOHE
)
from sklearn.decomposition import PCA
from scipy.stats import skew, kurtosis, entropy
import joblib
import os
import time
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
SUITE_ID = 99                              # OpenML-CC18
RESULTS_FILE = "performance_labels_v2.csv"
META_FEATURES_FILE = "meta_features_v2.csv"
MODEL_DIR = "meta_models_v2"
MAX_ROWS = 50000                           # Skip datasets larger than this

os.makedirs(MODEL_DIR, exist_ok=True)

# Constrained proxy model — forces the model to rely on feature engineering
# rather than brute-forcing the raw data (from DataCollector2)
PROXY_PARAMS = {
    'n_estimators': 200,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'max_depth': 5,            # CRITICAL: keep shallow!
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': -1,
    'n_jobs': -1
}

print("✅ Imports and configuration loaded.")

✅ Imports and configuration loaded.


## Step 1: Leakage-Free Evaluation Engine

Core principle: **every transformation is fit on the training fold and applied to the validation fold**.

This prevents data leakage from:
- Target encoding (target info leaks if computed on full data)
- Frequency encoding (frequency distribution leaks if computed globally)
- Imputation (median/mode leaks if computed on full data)
- Scaling / PCA (statistics leak if computed globally)

In [2]:
# ============================================================
# Leakage-Free Evaluation Engine
# ============================================================

def evaluate_method(X, y, method_fn=None):
    """
    5-Fold Stratified CV with leakage-free preprocessing.
    All transformations are applied WITHIN each fold.

    Args:
        X: Feature DataFrame
        y: Target Series (integer-encoded)
        method_fn: callable(X_tr, X_va, y_tr) -> (X_tr, X_va), or None for baseline

    Returns:
        (mean_accuracy, std_accuracy) or (None, None) on failure
    """
    # Use Stratified if possible, fallback to regular KFold
    try:
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(kf.split(X, y))
    except ValueError:
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(kf.split(X))

    scores = []

    for train_idx, val_idx in splits:
        # Reset index for clean concat later (OneHot, PCA add columns)
        X_tr = X.iloc[train_idx].copy().reset_index(drop=True)
        X_va = X.iloc[val_idx].copy().reset_index(drop=True)
        y_tr = y.iloc[train_idx].copy().reset_index(drop=True)
        y_va = y.iloc[val_idx].copy().reset_index(drop=True)

        # --- Apply method-specific transformation (within fold) ---
        if method_fn is not None:
            try:
                X_tr, X_va = method_fn(X_tr, X_va, y_tr)
            except Exception:
                return None, None

        # --- Encode remaining categorical columns (train-only mapping) ---
        cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
        for c in cat_cols:
            uniques = X_tr[c].astype(str).unique()
            mapping = {v: i for i, v in enumerate(uniques)}
            X_tr[c] = X_tr[c].astype(str).map(mapping).fillna(-1).astype(float)
            X_va[c] = X_va[c].astype(str).map(mapping).fillna(-1).astype(float)

        # --- Ensure all columns are numeric ---
        for c in X_tr.columns:
            if not np.issubdtype(X_tr[c].dtype, np.number):
                X_tr[c] = pd.to_numeric(X_tr[c], errors='coerce')
                X_va[c] = pd.to_numeric(X_va[c], errors='coerce')

        # --- Train with early stopping (constrained proxy model) ---
        try:
            model = lgb.LGBMClassifier(**PROXY_PARAMS)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
            )
            preds = model.predict(X_va, num_iteration=model.best_iteration_)
            scores.append(accuracy_score(y_va, preds))
        except Exception:
            return None, None

    return np.mean(scores), np.std(scores)


# ============================================================
# Preprocessing Methods (All leakage-free)
# Each function: (X_tr, X_va, y_tr) -> (X_tr, X_va)
# ============================================================

def method_frequency_encoding(X_tr, X_va, y_tr):
    """Replace categorical columns with frequency (computed from train only)."""
    cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
    for c in cat_cols:
        freq = X_tr[c].astype(str).value_counts(normalize=True)
        X_tr[c] = X_tr[c].astype(str).map(freq).fillna(0).astype(float)
        X_va[c] = X_va[c].astype(str).map(freq).fillna(0).astype(float)
    return X_tr, X_va


def method_target_encoding(X_tr, X_va, y_tr):
    """Smoothed target encoding (Bayesian) for categorical columns."""
    cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
    global_mean = y_tr.mean()
    m = 10  # smoothing factor
    for c in cat_cols:
        col_str = X_tr[c].astype(str)
        agg = y_tr.groupby(col_str).agg(['count', 'mean'])
        smooth = (agg['count'] * agg['mean'] + m * global_mean) / (agg['count'] + m)
        X_tr[c] = X_tr[c].astype(str).map(smooth).fillna(global_mean).astype(float)
        X_va[c] = X_va[c].astype(str).map(smooth).fillna(global_mean).astype(float)
    return X_tr, X_va


def method_onehot_encoding(X_tr, X_va, y_tr):
    """One-hot encode low-cardinality (<=10 unique) categorical columns."""
    cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
    low_card = [c for c in cat_cols if X_tr[c].nunique() <= 10]
    if len(low_card) == 0:
        return X_tr, X_va

    ohe = SklearnOHE(sparse_output=False, handle_unknown='ignore', drop='if_binary')
    tr_ohe = ohe.fit_transform(X_tr[low_card].astype(str))
    va_ohe = ohe.transform(X_va[low_card].astype(str))
    ohe_names = ohe.get_feature_names_out(low_card)

    X_tr = X_tr.drop(columns=low_card)
    X_va = X_va.drop(columns=low_card)
    X_tr = pd.concat([X_tr, pd.DataFrame(tr_ohe, columns=ohe_names)], axis=1)
    X_va = pd.concat([X_va, pd.DataFrame(va_ohe, columns=ohe_names)], axis=1)
    return X_tr, X_va


def method_standard_scaler(X_tr, X_va, y_tr):
    """Standardize numerical columns (fit on train only)."""
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) == 0:
        return X_tr, X_va
    # Fill NaN with train median before scaling
    for c in num_cols:
        med = X_tr[c].median()
        X_tr[c] = X_tr[c].fillna(med)
        X_va[c] = X_va[c].fillna(med)
    scaler = StandardScaler()
    X_tr[num_cols] = scaler.fit_transform(X_tr[num_cols])
    X_va[num_cols] = scaler.transform(X_va[num_cols])
    return X_tr, X_va


def method_log_transform(X_tr, X_va, y_tr):
    """Log-transform highly skewed (|skew|>1) numerical columns."""
    num_cols = X_tr.select_dtypes(include=[np.number]).columns
    skewed = [c for c in num_cols
              if len(X_tr[c].dropna()) > 0 and abs(skew(X_tr[c].dropna())) > 1]
    if len(skewed) == 0:
        return X_tr, X_va
    for c in skewed:
        offset = abs(X_tr[c].min()) + 1 if X_tr[c].min() <= 0 else 0
        X_tr[c] = np.log1p(X_tr[c] + offset)
        X_va[c] = np.log1p(X_va[c] + offset)
    return X_tr, X_va


def method_yeo_johnson(X_tr, X_va, y_tr):
    """Yeo-Johnson power transform for numerical columns."""
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) == 0:
        return X_tr, X_va
    for c in num_cols:
        med = X_tr[c].median()
        X_tr[c] = X_tr[c].fillna(med)
        X_va[c] = X_va[c].fillna(med)
    pt = PowerTransformer(method='yeo-johnson')
    X_tr[num_cols] = pt.fit_transform(X_tr[num_cols])
    X_va[num_cols] = pt.transform(X_va[num_cols])
    return X_tr, X_va


def method_impute_median(X_tr, X_va, y_tr):
    """Impute: median for numeric, mode for categorical (from train)."""
    for c in X_tr.select_dtypes(include=[np.number]).columns:
        if X_tr[c].isna().any() or X_va[c].isna().any():
            med = X_tr[c].median()
            X_tr[c] = X_tr[c].fillna(med)
            X_va[c] = X_va[c].fillna(med)
    for c in X_tr.select_dtypes(include=['object', 'category', 'bool']).columns:
        if X_tr[c].isna().any() or X_va[c].isna().any():
            mode_vals = X_tr[c].mode()
            fill = mode_vals.iloc[0] if len(mode_vals) > 0 else 'missing'
            X_tr[c] = X_tr[c].fillna(fill)
            X_va[c] = X_va[c].fillna(fill)
    return X_tr, X_va


def method_missing_indicator(X_tr, X_va, y_tr):
    """Add binary indicator columns for columns with missing values."""
    cols_with_na = [c for c in X_tr.columns
                    if X_tr[c].isna().any() or X_va[c].isna().any()]
    if len(cols_with_na) == 0:
        return X_tr, X_va
    for c in cols_with_na:
        X_tr[f'{c}_missing'] = X_tr[c].isna().astype(int)
        X_va[f'{c}_missing'] = X_va[c].isna().astype(int)
    return X_tr, X_va


def method_arithmetic_interactions(X_tr, X_va, y_tr):
    """Product interactions of top-5 highest-variance numerical columns."""
    num_cols = X_tr.select_dtypes(include=[np.number]).columns
    if len(num_cols) < 2:
        return X_tr, X_va
    # Fill NaN with train median first
    for c in num_cols:
        med = X_tr[c].median()
        X_tr[c] = X_tr[c].fillna(med)
        X_va[c] = X_va[c].fillna(med)
    top = X_tr[num_cols].var().sort_values(ascending=False).head(5).index
    for i, a in enumerate(top):
        for b in top[i+1:]:
            name = f'{a}_x_{b}'
            X_tr[name] = X_tr[a] * X_tr[b]
            X_va[name] = X_va[a] * X_va[b]
    return X_tr, X_va


def method_pca_95(X_tr, X_va, y_tr):
    """PCA on numerical columns, retaining 95% variance."""
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 3:
        return X_tr, X_va
    for c in num_cols:
        med = X_tr[c].median()
        X_tr[c] = X_tr[c].fillna(med)
        X_va[c] = X_va[c].fillna(med)
    scaler = StandardScaler()
    tr_scaled = scaler.fit_transform(X_tr[num_cols])
    va_scaled = scaler.transform(X_va[num_cols])
    pca = PCA(n_components=0.95, random_state=42)
    tr_pca = pca.fit_transform(tr_scaled)
    va_pca = pca.transform(va_scaled)
    X_tr = X_tr.drop(columns=num_cols)
    X_va = X_va.drop(columns=num_cols)
    pca_names = [f'PC_{i}' for i in range(tr_pca.shape[1])]
    X_tr = pd.concat([X_tr, pd.DataFrame(tr_pca, columns=pca_names)], axis=1)
    X_va = pd.concat([X_va, pd.DataFrame(va_pca, columns=pca_names)], axis=1)
    return X_tr, X_va


# ============================================================
# Method Registry & Applicability Check
# ============================================================

ALL_METHODS = {
    'FrequencyEncoding':        method_frequency_encoding,
    'TargetEncoding':           method_target_encoding,
    'OneHotEncoding':           method_onehot_encoding,
    'StandardScaler':           method_standard_scaler,
    'LogTransform':             method_log_transform,
    'YeoJohnson':               method_yeo_johnson,
    'ImputeMedian':             method_impute_median,
    'MissingIndicator':         method_missing_indicator,
    'ArithmeticInteractions':   method_arithmetic_interactions,
    'PCA_95':                   method_pca_95,
}


def get_applicable_methods(X):
    """
    Determine which methods are applicable to this dataset.
    Returns dict {method_name: method_fn}.
    """
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns
    num_cols = X.select_dtypes(include=[np.number]).columns
    has_missing = X.isna().any().any()
    has_cat = len(cat_cols) > 0
    has_num = len(num_cols) > 0
    has_low_card_cat = any(X[c].nunique() <= 10 for c in cat_cols) if has_cat else False
    has_skewed = False
    if has_num:
        for c in num_cols:
            clean = X[c].dropna()
            if len(clean) > 2 and abs(skew(clean)) > 1:
                has_skewed = True
                break

    applicable = {}
    if has_cat:
        applicable['FrequencyEncoding'] = method_frequency_encoding
        applicable['TargetEncoding'] = method_target_encoding
    if has_low_card_cat:
        applicable['OneHotEncoding'] = method_onehot_encoding
    if has_num:
        applicable['StandardScaler'] = method_standard_scaler
        applicable['YeoJohnson'] = method_yeo_johnson
    if has_skewed:
        applicable['LogTransform'] = method_log_transform
    if has_missing:
        applicable['ImputeMedian'] = method_impute_median
        applicable['MissingIndicator'] = method_missing_indicator
    if has_num and len(num_cols) >= 2:
        applicable['ArithmeticInteractions'] = method_arithmetic_interactions
    if has_num and len(num_cols) >= 3:
        applicable['PCA_95'] = method_pca_95

    return applicable


print(f"✅ Evaluation engine loaded. {len(ALL_METHODS)} methods available.")
print(f"   Methods: {list(ALL_METHODS.keys())}")

✅ Evaluation engine loaded. 10 methods available.
   Methods: ['FrequencyEncoding', 'TargetEncoding', 'OneHotEncoding', 'StandardScaler', 'LogTransform', 'YeoJohnson', 'ImputeMedian', 'MissingIndicator', 'ArithmeticInteractions', 'PCA_95']


## Step 2: Meta-Feature Extraction (70 features)

Dataset-level meta-features that describe the characteristics of a dataset:
- **Generic**: n_instances, n_attributes, dimensionality, missing values, class info
- **Continuous**: statistics of means, stds, skewness, kurtosis across numerical columns
- **Categorical**: statistics of entropy, mutual info, distinct values across categorical columns

In [3]:
# ============================================================
# Meta-Feature Extraction (70 features)
# ============================================================

def _get_quartiles(series):
    if len(series) == 0:
        return 0, 0, 0
    return series.quantile([0.25, 0.5, 0.75]).tolist()


def _calc_stats(series, prefix):
    """Min, Mean, Max, Std, Q1, Q2, Q3 for a series."""
    if len(series) == 0:
        return {f'Min_{prefix}': 0, f'Mean_{prefix}': 0, f'Max_{prefix}': 0,
                f'Std_{prefix}': 0, f'Q1_{prefix}': 0, f'Q2_{prefix}': 0, f'Q3_{prefix}': 0}
    q1, q2, q3 = _get_quartiles(series)
    return {
        f'Min_{prefix}': series.min(),
        f'Mean_{prefix}': series.mean(),
        f'Max_{prefix}': series.max(),
        f'Std_{prefix}': series.std(ddof=1) if len(series) > 1 else 0,
        f'Q1_{prefix}': q1, f'Q2_{prefix}': q2, f'Q3_{prefix}': q3
    }


def extract_meta_features(X, y, problem_type='classification'):
    """
    Extract 70 meta-features from a dataset.

    Args:
        X: Feature DataFrame (without target)
        y: Target Series
        problem_type: 'classification' or 'regression'

    Returns:
        dict of meta-features
    """
    f = {}  # features dict
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    n_inst = len(X)
    n_attr = len(X.columns)

    # ---- Generic Features ----
    f['Number_of_Instances'] = n_inst
    f['Number_of_Attributes'] = n_attr
    f['Dimensionality'] = n_attr / n_inst if n_inst > 0 else 0

    missing_total = X.isnull().sum().sum()
    f['Number_of_Missing_Values'] = missing_total
    f['Percentage_of_Missing_Values'] = missing_total / (n_inst * n_attr) if n_inst * n_attr > 0 else 0

    rows_missing = X.isnull().any(axis=1).sum()
    f['Number_of_Instances_with_Missing_Values'] = rows_missing
    f['Percentage_of_Instances_with_Missing_Values'] = rows_missing / n_inst if n_inst > 0 else 0

    # ---- Class Features ----
    if y is not None and len(y) > 0 and problem_type != 'regression':
        cc = y.value_counts()
        f['Number_of_Classes'] = len(cc)
        probs = cc / n_inst
        f['Class_Entropy'] = entropy(probs, base=2)
        f['Minority_Class_Size'] = cc.min()
        f['Majority_Class_Size'] = cc.max()
        f['Minority_Class_Percentage'] = cc.min() / n_inst
        f['Majority_Class_Percentage'] = cc.max() / n_inst
    else:
        for k in ['Number_of_Classes', 'Class_Entropy', 'Minority_Class_Size',
                   'Majority_Class_Size', 'Minority_Class_Percentage', 'Majority_Class_Percentage']:
            f[k] = 0

    # ---- Continuous Attribute Statistics ----
    f['Number_of_Continuous_Attributes'] = len(num_cols)
    f['Percentage_of_Continuous_Attributes'] = len(num_cols) / n_attr if n_attr > 0 else 0

    if len(num_cols) > 0:
        X_num = X[num_cols]
        means  = X_num.mean()
        stds   = X_num.std()
        skews  = X_num.apply(lambda s: skew(s.dropna()) if len(s.dropna()) > 2 else 0)
        kurts  = X_num.apply(lambda s: kurtosis(s.dropna()) if len(s.dropna()) > 2 else 0)
        f.update(_calc_stats(means,  'Means_of_Continuous_Attributes'))
        f.update(_calc_stats(stds,   'Std_of_Continuous_Attributes'))
        f.update(_calc_stats(skews,  'Skewness_of_Continuous_Attributes'))
        f.update(_calc_stats(kurts,  'Kurtosis_of_Continuous_Attributes'))
    else:
        for stat in ['Means', 'Std', 'Skewness', 'Kurtosis']:
            f.update(_calc_stats(pd.Series([], dtype=float), f'{stat}_of_Continuous_Attributes'))

    # ---- Categorical & Binary Attribute Statistics ----
    binary_cols = [c for c in X.columns if X[c].nunique() == 2]
    f['Number_of_Categorical_Attributes'] = len(cat_cols)
    f['Number_of_Binary_Attributes'] = len(binary_cols)
    f['Percentage_of_Categorical_Attributes'] = len(cat_cols) / n_attr if n_attr > 0 else 0
    f['Percentage_of_Binary_Attributes'] = len(binary_cols) / n_attr if n_attr > 0 else 0

    if len(cat_cols) > 0:
        entropies, mutual_infos, distinct_vals = [], [], []
        for c in cat_cols:
            vc = X[c].astype(str).value_counts(normalize=True)
            entropies.append(entropy(vc, base=2))
            distinct_vals.append(X[c].nunique())
            mutual_infos.append(0)  # simplified for speed
        f.update(_calc_stats(pd.Series(entropies),     'Attribute_Entropy'))
        f.update(_calc_stats(pd.Series(mutual_infos),  'Mutual_Information'))
        f.update(_calc_stats(pd.Series(distinct_vals, dtype=float), 'Attribute_Distinct_Values'))
        f['Equivalent_Number_of_Attributes'] = 0
        f['Noise_to_Signal_Ratio'] = 0
    else:
        for stat in ['Attribute_Entropy', 'Mutual_Information', 'Attribute_Distinct_Values']:
            f.update(_calc_stats(pd.Series([], dtype=float), stat))
        f['Equivalent_Number_of_Attributes'] = 0
        f['Noise_to_Signal_Ratio'] = 0

    return f


# Quick test
_test_X = pd.DataFrame({'a': [1,2,3,4,5], 'b': ['x','y','x','y','z']})
_test_y = pd.Series([0, 1, 0, 1, 0])
_feats = extract_meta_features(_test_X, _test_y)
print(f"✅ Meta-feature extraction loaded. Produces {len(_feats)} features.")

✅ Meta-feature extraction loaded. Produces 70 features.


## Step 3: Collect Performance Labels from CC18

For each dataset in OpenML-CC18:
1. Download X, y from OpenML
2. Extract 70 meta-features
3. Evaluate Baseline (no preprocessing) using leakage-free 5-Fold CV
4. Evaluate each applicable preprocessing method
5. Record score, delta, and method for meta-model training

**Resume-capable**: skips datasets already saved in CSV.

In [4]:
# ============================================================
# Data Collection from CC18
# ============================================================

print(f"🔍 Loading OpenML-CC18 (Suite {SUITE_ID})...")
suite = openml.study.get_suite(SUITE_ID)
task_ids = list(suite.tasks)
print(f"   Found {len(task_ids)} tasks.")

# Load existing results for resume
if os.path.exists(RESULTS_FILE):
    existing_labels = pd.read_csv(RESULTS_FILE)
    processed_ids = set(existing_labels['dataset_id'].unique())
    all_labels = existing_labels.to_dict('records')
    print(f"   Resuming: {len(processed_ids)} datasets already done.")
else:
    all_labels = []
    processed_ids = set()

if os.path.exists(META_FEATURES_FILE):
    existing_meta = pd.read_csv(META_FEATURES_FILE)
    all_meta = existing_meta.to_dict('records')
else:
    all_meta = []

total = len(task_ids)
t_start = time.time()

for idx, task_id in enumerate(task_ids):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{total}] Task {task_id}")

    try:
        # --- Load dataset ---
        task = openml.tasks.get_task(task_id)
        dataset = task.get_dataset()
        dataset_id = dataset.id
        dataset_name = dataset.name
        target_name = task.target_name

        if dataset_id in processed_ids:
            print(f"   ⏭️  {dataset_name} — already processed")
            continue

        X, y_raw, _, _ = dataset.get_data(target=target_name)

        # --- Basic checks ---
        if len(X) > MAX_ROWS:
            print(f"   ⚠️  {dataset_name} — too large ({len(X)} rows), skipping")
            continue
        if len(X) < 50:
            print(f"   ⚠️  {dataset_name} — too small ({len(X)} rows), skipping")
            continue

        # --- Encode target ---
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y_raw.astype(str)), name='target')

        # Drop datetime columns (if any)
        dt_cols = X.select_dtypes(include=['datetime64', 'datetime64[ns]']).columns
        if len(dt_cols) > 0:
            X = X.drop(columns=dt_cols)

        n_classes = y.nunique()
        print(f"   📊 {dataset_name} | {X.shape[0]} rows x {X.shape[1]} cols | {n_classes} classes")

        # --- Extract meta-features ---
        mf = extract_meta_features(X, y)
        mf['dataset_id'] = dataset_id
        mf['dataset_name'] = dataset_name
        all_meta.append(mf)

        # --- Evaluate Baseline ---
        t0 = time.time()
        base_score, base_std = evaluate_method(X, y, method_fn=None)
        t_base = time.time() - t0

        if base_score is None:
            print(f"   ❌ Baseline evaluation failed, skipping dataset")
            continue

        all_labels.append({
            'dataset_id': dataset_id, 'dataset_name': dataset_name,
            'method': 'Baseline', 'score': base_score,
            'score_std': base_std, 'time_seconds': t_base
        })
        print(f"   ✅ Baseline: {base_score:.4f} (±{base_std:.4f}) [{t_base:.1f}s]")

        # --- Evaluate each applicable method ---
        applicable = get_applicable_methods(X)
        print(f"   🔧 Testing {len(applicable)} methods...")

        for method_name, method_fn in applicable.items():
            t0 = time.time()
            score, std = evaluate_method(X, y, method_fn)
            elapsed = time.time() - t0

            if score is not None:
                delta = score - base_score
                all_labels.append({
                    'dataset_id': dataset_id, 'dataset_name': dataset_name,
                    'method': method_name, 'score': score,
                    'score_std': std, 'time_seconds': elapsed
                })
                icon = '✅' if delta >= 0 else '❌'
                print(f"      {icon} {method_name:25s} | Acc={score:.4f} | Δ={delta:+.4f} [{elapsed:.1f}s]")
            else:
                print(f"      ⚠️  {method_name:25s} | FAILED [{elapsed:.1f}s]")

        # --- Save intermediate results ---
        processed_ids.add(dataset_id)
        pd.DataFrame(all_labels).to_csv(RESULTS_FILE, index=False)
        pd.DataFrame(all_meta).to_csv(META_FEATURES_FILE, index=False)

        elapsed_total = time.time() - t_start
        print(f"   💾 Saved. Total time: {elapsed_total/60:.1f} min")

    except Exception as e:
        print(f"   ❌ Error: {e}")
        continue

# --- Final Summary ---
df_labels = pd.DataFrame(all_labels)
df_meta = pd.DataFrame(all_meta)
print(f"\n{'='*60}")
print(f"✅ Data collection complete!")
print(f"   Labels:        {len(df_labels)} rows ({df_labels['dataset_id'].nunique()} datasets)")
print(f"   Meta-features: {len(df_meta)} datasets x {len(df_meta.columns)} columns")
print(f"   Methods tested per dataset (avg): {len(df_labels) / df_labels['dataset_id'].nunique():.1f}")

🔍 Loading OpenML-CC18 (Suite 99)...
   Found 72 tasks.
   Resuming: 66 datasets already done.

[1/72] Task 3
   ⏭️  kr-vs-kp — already processed

[2/72] Task 6
   ⏭️  letter — already processed

[3/72] Task 11
   ⏭️  balance-scale — already processed

[4/72] Task 12
   ⏭️  mfeat-factors — already processed

[5/72] Task 14
   ⏭️  mfeat-fourier — already processed

[6/72] Task 15
   ⏭️  breast-w — already processed

[7/72] Task 16
   ⏭️  mfeat-karhunen — already processed

[8/72] Task 18
   ⏭️  mfeat-morphological — already processed

[9/72] Task 22
   ⏭️  mfeat-zernike — already processed

[10/72] Task 23
   ⏭️  cmc — already processed

[11/72] Task 28
   ⏭️  optdigits — already processed

[12/72] Task 29
   ⏭️  credit-approval — already processed

[13/72] Task 31
   ⏭️  credit-g — already processed

[14/72] Task 32
   ⏭️  pendigits — already processed

[15/72] Task 37
   ⏭️  diabetes — already processed

[16/72] Task 43
   ⏭️  spambase — already processed

[17/72] Task 45
   ⏭️  splice

## Step 4: Train Meta-Model Regressors

For each preprocessing method:
1. Compute **gain = method_score − baseline_score** for each dataset
2. Train a LightGBM regressor: **X = 70 meta-features, y = gain**
3. Evaluate with 5-Fold CV using **Win Accuracy** (correctly predict gain > 0 or gain < 0)
4. Save model to disk

In [5]:
# ============================================================
# Meta-Model Training (Regression)
# ============================================================

# Load collected data
df_labels = pd.read_csv(RESULTS_FILE)
df_meta   = pd.read_csv(META_FEATURES_FILE)

print(f"📂 Loaded {len(df_labels)} labels, {len(df_meta)} datasets")

# Compute gain
baseline = df_labels[df_labels['method'] == 'Baseline'].set_index('dataset_id')['score']
df_labels['baseline_score'] = df_labels['dataset_id'].map(baseline)
df_labels['gain'] = df_labels['score'] - df_labels['baseline_score']
df_labels = df_labels.dropna(subset=['gain'])

# Merge with meta-features
full_df = pd.merge(df_labels, df_meta, on='dataset_id', how='inner')

# Identify feature columns (exclude metadata)
exclude = {'dataset_id', 'dataset_name', 'dataset_name_x', 'dataset_name_y',
           'method', 'score', 'score_std', 'time_seconds', 'baseline_score', 'gain'}
feature_cols = [c for c in full_df.select_dtypes(include=[np.number]).columns
                if c not in exclude]

print(f"🧠 Meta-features for training: {len(feature_cols)}")
print(f"📊 Total samples: {len(full_df)} ({full_df['dataset_id'].nunique()} datasets)")

# Train one regressor per method
methods = [m for m in full_df['method'].unique() if m != 'Baseline']
results = []

print(f"\n🏋️ Training {len(methods)} regressors...")
print("-" * 70)

for method in methods:
    mdf = full_df[full_df['method'] == method]

    if len(mdf) < 5:
        print(f"   ⚠️  {method:25s} | Skipped (only {len(mdf)} samples)")
        continue

    X_m = mdf[feature_cols].copy()
    y_m = mdf['gain'].copy()

    # Fill any NaN in meta-features
    X_m = X_m.fillna(0)

    model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1)

    n_splits = min(5, len(mdf))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_pred_cv = cross_val_predict(model, X_m, y_m, cv=kf)

    # Final fit on all data
    model.fit(X_m, y_m)
    joblib.dump(model, f"{MODEL_DIR}/regressor_{method}.pkl")

    rmse = np.sqrt(mean_squared_error(y_m, y_pred_cv))
    win_acc = np.mean((y_m > 0).values == (y_pred_cv > 0))

    results.append({
        'Method': method,
        'Samples': len(mdf),
        'RMSE': round(rmse, 6),
        'Win_Accuracy': round(win_acc, 4),
        'Avg_Gain': round(y_m.mean(), 6)
    })
    print(f"   ✅ {method:25s} | N={len(mdf):3d} | WinAcc={win_acc:.2%} | AvgGain={y_m.mean():+.4f}")

# Save feature column names for inference
joblib.dump(feature_cols, f"{MODEL_DIR}/feature_cols.pkl")

# Summary
res_df = pd.DataFrame(results).sort_values('Win_Accuracy', ascending=False)
print(f"\n{'='*70}")
print("📊 Meta-Model Training Summary (sorted by Win Accuracy):")
print(res_df.to_string(index=False))
res_df.to_csv("meta_model_summary_v2.csv", index=False)

📂 Loaded 435 labels, 66 datasets
🧠 Meta-features for training: 70
📊 Total samples: 435 (66 datasets)

🏋️ Training 10 regressors...
----------------------------------------------------------------------
   ✅ FrequencyEncoding         | N= 22 | WinAcc=63.64% | AvgGain=-0.0022
   ✅ TargetEncoding            | N= 22 | WinAcc=40.91% | AvgGain=-0.0002
   ✅ OneHotEncoding            | N= 22 | WinAcc=40.91% | AvgGain=-0.0007
   ✅ StandardScaler            | N= 59 | WinAcc=54.24% | AvgGain=+0.0005
   ✅ YeoJohnson                | N= 59 | WinAcc=38.98% | AvgGain=+0.0004
   ✅ LogTransform              | N= 52 | WinAcc=30.77% | AvgGain=+0.0003
   ✅ ArithmeticInteractions    | N= 58 | WinAcc=51.72% | AvgGain=+0.0021
   ✅ PCA_95                    | N= 57 | WinAcc=82.46% | AvgGain=-0.0216
   ✅ ImputeMedian              | N=  9 | WinAcc=44.44% | AvgGain=-0.0012
   ✅ MissingIndicator          | N=  9 | WinAcc=55.56% | AvgGain=-0.0001

📊 Meta-Model Training Summary (sorted by Win Accuracy):
           

## Step 5: Inference — Recommend Preprocessing for a New Dataset

Given a new dataset:
1. Extract 70 meta-features
2. Load all trained regressors
3. Predict gain for each preprocessing method
4. Recommend methods with positive predicted gain

In [6]:
# ============================================================
# Inference Pipeline
# ============================================================

def recommend_preprocessing(df, target_col=None, problem_type='classification', top_k=5):
    """
    Recommend preprocessing methods for a new dataset.

    Args:
        df: Input DataFrame (with or without target column)
        target_col: Name of the target column
        problem_type: 'classification' or 'regression'
        top_k: Number of top recommendations to display

    Returns:
        DataFrame with predicted gains for each method
    """
    print("🤖 Meta-Model Inference (Regression)...")

    if not os.path.exists(MODEL_DIR):
        print(f"❌ Model directory '{MODEL_DIR}' not found. Train models first (Step 4).")
        return None

    # 1. Separate X and y
    if target_col and target_col in df.columns:
        X = df.drop(columns=[target_col])
        y = df[target_col]
    else:
        X = df
        y = pd.Series(dtype=float)

    # 2. Extract meta-features
    print("   Extracting meta-features...")
    mf = extract_meta_features(X, y, problem_type)
    mf_df = pd.DataFrame([mf])
    print(f"   ✅ Extracted {len(mf)} features")

    # 3. Load feature column names
    feature_cols = joblib.load(f"{MODEL_DIR}/feature_cols.pkl")

    # Align features (fill missing with 0)
    for c in feature_cols:
        if c not in mf_df.columns:
            mf_df[c] = 0
    X_meta = mf_df[feature_cols]

    # 4. Predict with each regressor
    predictions = []
    for fname in os.listdir(MODEL_DIR):
        if not fname.startswith('regressor_') or not fname.endswith('.pkl'):
            continue
        method = fname.replace('regressor_', '').replace('.pkl', '')
        try:
            model = joblib.load(os.path.join(MODEL_DIR, fname))
            # Use model's own feature names if available
            if hasattr(model, 'feature_name_'):
                model_feats = model.feature_name_
                for c in model_feats:
                    if c not in mf_df.columns:
                        mf_df[c] = 0
                X_input = mf_df[model_feats]
            else:
                X_input = X_meta
            pred = model.predict(X_input)[0]
            predictions.append({
                'Method': method,
                'Predicted_Gain': pred,
                'Recommended': pred > 0
            })
        except Exception as e:
            print(f"   ⚠️ Error with {method}: {e}")

    if not predictions:
        print("❌ No predictions. Check if models exist.")
        return None

    result = pd.DataFrame(predictions).sort_values('Predicted_Gain', ascending=False)

    # 5. Display
    print(f"\n📊 Top-{top_k} Preprocessing Recommendations:")
    print("-" * 55)
    for _, row in result.head(top_k).iterrows():
        icon = '✅' if row['Recommended'] else '❌'
        print(f"   {icon} {row['Method']:25s} | Gain: {row['Predicted_Gain']:+.4f}")
    print("-" * 55)

    rec = result[result['Recommended']]
    if len(rec) > 0:
        print(f"\n🏆 Top Recommended: {rec.iloc[0]['Method']}")
    else:
        print("\n⚠️ No method predicted to improve performance. Use Baseline.")

    return result


print("✅ Inference pipeline loaded.")

✅ Inference pipeline loaded.


## Demo: Run Inference on a Test Dataset

In [7]:
# Create a synthetic test dataset for demonstration
np.random.seed(42)
df_demo = pd.DataFrame({
    'age':       np.random.randint(18, 80, 500),
    'income':    np.random.exponential(50000, 500),
    'score':     np.random.normal(100, 15, 500),
    'city':      np.random.choice(['Berlin', 'Munich', 'Hamburg', 'Frankfurt'], 500),
    'category':  np.random.choice(['A', 'B', 'C', 'D', 'E'], 500),
    'has_debt':  np.random.choice([0, 1], 500),
    'target':    np.random.randint(0, 2, 500)
})
# Add some missing values
df_demo.loc[df_demo.sample(50, random_state=1).index, 'income'] = np.nan
df_demo.loc[df_demo.sample(30, random_state=2).index, 'city'] = np.nan

print("📋 Demo dataset:")
print(f"   Shape: {df_demo.shape}")
print(f"   Missing values: {df_demo.isnull().sum().sum()}")
print(f"   Columns: {list(df_demo.columns)}\n")

# Run inference
result = recommend_preprocessing(df_demo, target_col='target')

📋 Demo dataset:
   Shape: (500, 7)
   Missing values: 80
   Columns: ['age', 'income', 'score', 'city', 'category', 'has_debt', 'target']

🤖 Meta-Model Inference (Regression)...
   Extracting meta-features...
   ✅ Extracted 70 features

📊 Top-5 Preprocessing Recommendations:
-------------------------------------------------------
   ✅ ArithmeticInteractions    | Gain: +0.0081
   ❌ MissingIndicator          | Gain: -0.0001
   ❌ TargetEncoding            | Gain: -0.0002
   ❌ YeoJohnson                | Gain: -0.0002
   ❌ LogTransform              | Gain: -0.0003
-------------------------------------------------------

🏆 Top Recommended: ArithmeticInteractions


## Step 6: Test with Real Datasets

Run inference on 8 real-world test datasets from the `testset/` directory.
These datasets were NOT used during meta-model training (CC18 only), so they serve as
a genuine out-of-distribution test of the meta-model's generalization ability.

| Dataset | Rows | Cols | Classes | Notable |
|---------|------|------|---------|--------|
| maternal_health_risk | 1,014 | 7 | 3 | Small, clean, numeric-heavy |
| MIC | 1,699 | 112 | 8 | High dimensionality, massive missing values |
| splice | 3,190 | 61 | 3 | All categorical |
| hiva_agnostic | 3,845 | 1,618 | 3 | Ultra-high dimensionality |
| E-CommerceShippingData | 10,999 | 11 | 2 | Mixed types |
| online_shoppers_intention | 12,330 | 18 | 2 | Mixed types |
| Amazon_employee_access | 32,769 | 10 | 2 | All numeric (IDs), imbalanced |
| APSFailure | 76,000 | 171 | 2 | Large, massive missing values |

In [8]:
# ============================================================
# Test with Real Datasets from testset/ folder
# ============================================================

TESTSET_DIR = "testset"

TEST_DATASETS = [
    {"file": "maternal_health_risk.csv",      "target": "RiskLevel"},
    {"file": "MIC.csv",                       "target": "LET_IS"},
    {"file": "splice.csv",                    "target": "SiteType"},
    {"file": "hiva_agnostic.csv",             "target": "CompoundActivity"},
    {"file": "E-CommereShippingData.csv",     "target": "ArrivedLate"},
    {"file": "online_shoppers_intention.csv", "target": "Revenue"},
    {"file": "Amazon_employee_access.csv",    "target": "ResourceApproved"},
    {"file": "APSFailure.csv",                "target": "AirPressureSystemFailure"},
]

all_test_results = []

print("=" * 70)
print("  REAL-WORLD DATASET TEST (Out-of-Distribution)")
print(f"  Model dir: {MODEL_DIR}")
print("=" * 70)

for info in TEST_DATASETS:
    filepath = os.path.join(TESTSET_DIR, info["file"])
    target_col = info["target"]
    dataset_name = info["file"].replace(".csv", "")

    if not os.path.exists(filepath):
        print(f"\n--- {info['file']} --- FILE NOT FOUND, skipping")
        continue

    print(f"\n{'=' * 60}")
    print(f"Dataset: {dataset_name}")
    print("-" * 60)

    try:
        df = pd.read_csv(filepath)
        n_rows, n_cols = df.shape
        n_missing = df.isna().sum().sum()
        n_classes = df[target_col].nunique() if target_col in df.columns else "?"

        print(f"  Shape:    {n_rows} rows x {n_cols} cols")
        print(f"  Target:   '{target_col}' ({n_classes} classes)")
        print(f"  Missing:  {n_missing} values ({n_missing / (n_rows * n_cols) * 100:.1f}%)")

        # Run inference
        result = recommend_preprocessing(df, target_col=target_col, top_k=5)

        if result is not None:
            result["dataset"] = dataset_name
            all_test_results.append(result)

    except Exception as e:
        print(f"  ERROR: {e}")

# ============================================================
# Summary Table
# ============================================================
if all_test_results:
    print(f"\n\n{'=' * 70}")
    print("  SUMMARY: Top Recommendation per Dataset")
    print("=" * 70)
    print(f"{'Dataset':<35s} {'Top Method':<25s} {'Gain':>10s}")
    print("-" * 70)

    for res in all_test_results:
        ds = res["dataset"].iloc[0]
        top = res.iloc[0]
        icon = "\u2705" if top["Recommended"] else "\u274c"
        print(f"{icon} {ds:<33s} {top['Method']:<25s} {top['Predicted_Gain']:+.4f}")

    print("-" * 70)
    print(f"\nTotal datasets tested: {len(all_test_results)}")
else:
    print("\nNo test results. Make sure models are trained (Step 4) and testset/ folder exists.")

  REAL-WORLD DATASET TEST (Out-of-Distribution)
  Model dir: meta_models_v2

Dataset: maternal_health_risk
------------------------------------------------------------
  Shape:    1014 rows x 7 cols
  Target:   'RiskLevel' (3 classes)
  Missing:  0 values (0.0%)
🤖 Meta-Model Inference (Regression)...
   Extracting meta-features...
   ✅ Extracted 70 features

📊 Top-5 Preprocessing Recommendations:
-------------------------------------------------------
   ✅ ArithmeticInteractions    | Gain: +0.0012
   ✅ YeoJohnson                | Gain: +0.0012
   ❌ MissingIndicator          | Gain: -0.0001
   ❌ TargetEncoding            | Gain: -0.0002
   ❌ LogTransform              | Gain: -0.0002
-------------------------------------------------------

🏆 Top Recommended: ArithmeticInteractions

Dataset: MIC
------------------------------------------------------------
  Shape:    1699 rows x 112 cols
  Target:   'LET_IS' (8 classes)
  Missing:  15965 values (8.4%)
🤖 Meta-Model Inference (Regression)..

## Step 7: Ground-Truth Validation

The meta-model **predicts** gain, but does the recommended preprocessing **actually** improve accuracy?

For each testset dataset:
1. Run **Baseline** 5-fold CV (no preprocessing)
2. Run each **individually recommended** method (positive predicted gain) with 5-fold CV
3. Build a **combined pipeline** from compatible recommended methods and run 5-fold CV
4. Compare predicted vs actual gain, and measure **direction accuracy** (did the meta-model correctly predict whether a method helps or hurts?)

**Combined pipeline logic** — mutually exclusive groups, pick highest predicted gain from each:
- Imputation → Indicator → Scaling/Transform → Encoding → Feature Engineering

In [9]:
# ============================================================
# Ground-Truth Validation on Testset
# ============================================================

MAX_VAL_ROWS = 10000  # Sample large datasets for reasonable runtime

# Mutually exclusive method groups (pick best from each)
SCALING_GROUP = {'StandardScaler', 'RobustScaler', 'YeoJohnson', 'LogTransform', 'ClipOutliers'}
ENCODING_GROUP = {'FrequencyEncoding', 'TargetEncoding', 'OneHotEncoding'}
FEATURE_GROUP = {'ArithmeticInteractions', 'PCA_95'}
IMPUTE_GROUP = {'ImputeMedian'}
INDICATOR_GROUP = {'MissingIndicator'}

# Pipeline order: impute first, feature engineering last
PIPELINE_GROUPS = [
    ('imputation', IMPUTE_GROUP),
    ('indicator', INDICATOR_GROUP),
    ('scaling', SCALING_GROUP),
    ('encoding', ENCODING_GROUP),
    ('feature_eng', FEATURE_GROUP),
]


def build_combined_fn(positive_preds, method_registry):
    """
    Build a combined preprocessing function from positive-gain methods.
    Within each mutually exclusive group, pick the highest predicted gain.
    Apply in pipeline order.
    """
    selected = []
    for group_name, group_set in PIPELINE_GROUPS:
        candidates = [(m['Method'], m['Predicted_Gain'])
                      for m in positive_preds
                      if m['Method'] in group_set and m['Method'] in method_registry]
        if candidates:
            best_name = max(candidates, key=lambda x: x[1])[0]
            selected.append(best_name)

    if not selected:
        return None, []

    def combined(X_tr, X_va, y_tr):
        for name in selected:
            fn = method_registry[name]
            X_tr, X_va = fn(X_tr, X_va, y_tr)
        return X_tr, X_va

    return combined, selected


# ============================================================
# Run Validation
# ============================================================

print("=" * 75)
print("  GROUND-TRUTH VALIDATION ON TESTSET")
print("  Running actual 5-Fold CV to verify meta-model predictions")
print("=" * 75)

all_validation = []

for info in TEST_DATASETS:
    filepath = os.path.join(TESTSET_DIR, info["file"])
    target_col = info["target"]
    dataset_name = info["file"].replace(".csv", "")

    if not os.path.exists(filepath):
        print(f"\n--- {dataset_name} --- FILE NOT FOUND, skipping")
        continue

    print(f"\n{'=' * 70}")
    print(f"  {dataset_name}")
    print(f"{'=' * 70}")

    try:
        df = pd.read_csv(filepath)

        # Sample large datasets
        if len(df) > MAX_VAL_ROWS:
            print(f"  Sampling {MAX_VAL_ROWS} from {len(df)} rows for speed")
            df = df.sample(n=MAX_VAL_ROWS, random_state=42).reset_index(drop=True)

        # Separate X, y
        X = df.drop(columns=[target_col])
        y_raw = df[target_col]

        # Drop datetime columns
        dt_cols = X.select_dtypes(include=['datetime64', 'datetime64[ns]']).columns
        if len(dt_cols) > 0:
            X = X.drop(columns=dt_cols)

        # Encode target
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y_raw.astype(str)), name='target')

        print(f"  Shape: {X.shape[0]} x {X.shape[1]} | {y.nunique()} classes")

        # --- Get meta-model predictions ---
        mf = extract_meta_features(X, y)
        mf_df = pd.DataFrame([mf])
        feat_cols = joblib.load(f"{MODEL_DIR}/feature_cols.pkl")
        for c in feat_cols:
            if c not in mf_df.columns:
                mf_df[c] = 0

        predictions = []
        for fname in os.listdir(MODEL_DIR):
            if not fname.startswith('regressor_') or not fname.endswith('.pkl'):
                continue
            method = fname.replace('regressor_', '').replace('.pkl', '')
            mdl = joblib.load(os.path.join(MODEL_DIR, fname))
            if hasattr(mdl, 'feature_name_'):
                m_feats = mdl.feature_name_
                for c in m_feats:
                    if c not in mf_df.columns:
                        mf_df[c] = 0
                X_input = mf_df[m_feats]
            else:
                X_input = mf_df[feat_cols]
            pred = mdl.predict(X_input)[0]
            predictions.append({'Method': method, 'Predicted_Gain': pred})

        positive_preds = sorted(
            [p for p in predictions if p['Predicted_Gain'] > 0],
            key=lambda x: x['Predicted_Gain'], reverse=True
        )

        print(f"  Meta-model recommends {len(positive_preds)} methods with positive gain\n")

        # --- Baseline ---
        t0 = time.time()
        base_score, base_std = evaluate_method(X, y, method_fn=None)
        t_base = time.time() - t0

        if base_score is None:
            print(f"  Baseline FAILED, skipping dataset")
            continue

        print(f"  {'Method':<30s} {'Predicted':>10s} {'Actual':>10s} {'Match':>6s} {'Time':>7s}")
        print(f"  {'-' * 65}")
        print(f"  {'Baseline':<30s} {'---':>10s} {base_score:>10.4f} {'':>6s} {t_base:>6.1f}s")

        dataset_results = {
            'dataset': dataset_name,
            'baseline': base_score,
            'methods': {},
        }

        # --- Individual methods ---
        applicable = get_applicable_methods(X)

        for pred in positive_preds:
            method_name = pred['Method']
            predicted_gain = pred['Predicted_Gain']

            if method_name not in applicable:
                continue

            method_fn = applicable[method_name]
            t0 = time.time()
            score, std = evaluate_method(X, y, method_fn)
            elapsed = time.time() - t0

            if score is not None:
                actual_gain = score - base_score
                direction_match = (predicted_gain > 0) == (actual_gain > 0)
                mark = "\u2713" if direction_match else "\u2717"

                print(f"  {method_name:<30s} {predicted_gain:>+10.4f} {actual_gain:>+10.4f} {'  ' + mark:>6s} {elapsed:>6.1f}s")

                dataset_results['methods'][method_name] = {
                    'predicted_gain': predicted_gain,
                    'actual_gain': actual_gain,
                    'actual_score': score,
                    'direction_match': direction_match,
                }
            else:
                print(f"  {method_name:<30s} {predicted_gain:>+10.4f} {'FAILED':>10s} {'':>6s} {elapsed:>6.1f}s")

        # --- Combined pipeline ---
        combined_fn, selected_names = build_combined_fn(positive_preds, applicable)

        if combined_fn is not None and len(selected_names) > 1:
            t0 = time.time()
            combo_score, combo_std = evaluate_method(X, y, combined_fn)
            elapsed = time.time() - t0

            if combo_score is not None:
                combo_gain = combo_score - base_score
                print(f"  {'-' * 65}")
                print(f"  {'COMBINED PIPELINE':<30s} {'':>10s} {combo_gain:>+10.4f} {'':>6s} {elapsed:>6.1f}s")
                print(f"  Steps: {' -> '.join(selected_names)}")

                dataset_results['combined_gain'] = combo_gain
                dataset_results['combined_steps'] = selected_names

        all_validation.append(dataset_results)

    except Exception as e:
        print(f"  ERROR: {e}")
        continue

# ============================================================
# Final Summary
# ============================================================
if all_validation:
    print(f"\n\n{'=' * 75}")
    print("  VALIDATION SUMMARY")
    print(f"{'=' * 75}")
    print(f"  {'Dataset':<26s} {'Baseline':>9s} {'Best_Single':>13s} {'Combined':>10s} {'Direction':>10s}")
    print(f"  {'-' * 70}")

    total_correct = 0
    total_preds = 0

    for v in all_validation:
        ds = v['dataset']
        bl = v['baseline']

        # Best single method by actual gain
        if v['methods']:
            best = max(v['methods'].items(), key=lambda x: x[1]['actual_gain'])
            best_str = f"{best[1]['actual_gain']:+.4f}"
            for m_info in v['methods'].values():
                total_preds += 1
                if m_info['direction_match']:
                    total_correct += 1
        else:
            best_str = "N/A"

        combo_str = f"{v['combined_gain']:+.4f}" if 'combined_gain' in v else "N/A"

        ds_correct = sum(1 for m in v['methods'].values() if m['direction_match'])
        ds_total = len(v['methods'])
        dir_str = f"{ds_correct}/{ds_total}" if ds_total > 0 else "N/A"

        print(f"  {ds:<26s} {bl:>9.4f} {best_str:>13s} {combo_str:>10s} {dir_str:>10s}")

    print(f"  {'-' * 70}")
    if total_preds > 0:
        dir_acc = total_correct / total_preds * 100
        print(f"\n  Overall Direction Accuracy: {total_correct}/{total_preds} ({dir_acc:.1f}%)")
        print(f"  (meta-model correctly predicted whether a method helps or hurts)")
else:
    print("\nNo validation results.")

  GROUND-TRUTH VALIDATION ON TESTSET
  Running actual 5-Fold CV to verify meta-model predictions

  maternal_health_risk
  Shape: 1014 x 6 | 3 classes
  Meta-model recommends 2 methods with positive gain

  Method                          Predicted     Actual  Match    Time
  -----------------------------------------------------------------
  Baseline                              ---     0.8313          24.2s
  ArithmeticInteractions            +0.0012    +0.0079      ✓   21.1s
  YeoJohnson                        +0.0012    -0.0039      ✗   29.2s
  -----------------------------------------------------------------
  COMBINED PIPELINE                            +0.0138          28.2s
  Steps: YeoJohnson -> ArithmeticInteractions

  MIC
  Shape: 1699 x 111 | 8 classes
  Meta-model recommends 3 methods with positive gain

  Method                          Predicted     Actual  Match    Time
  -----------------------------------------------------------------
  Baseline                      

# Meta-Learning Pipeline v2 — Ground-Truth Validation Results

## Overview

We trained a meta-learning system on 66 OpenML-CC18 datasets to predict which
preprocessing methods would improve classification accuracy for unseen datasets.
The system was then validated on 8 real-world test datasets by actually running
5-fold cross-validation with the recommended methods.

## Key Metrics

- **Training data**: 66 CC18 datasets, 435 performance labels, 10 preprocessing methods
- **Meta-features**: 70 dataset-level descriptors (generic, continuous, categorical statistics)
- **Meta-model**: One LightGBM regressor per method, predicting gain over baseline
- **Evaluation metric**: Direction Accuracy — did the meta-model correctly predict
  whether a method helps (+gain) or hurts (−gain)?

## Validation Results on 8 Test Datasets

| Dataset                   | Baseline | Best Single | Combined | Direction Acc |
|---------------------------|----------|-------------|----------|---------------|
| maternal_health_risk      | 0.8313   | +0.0079     | +0.0138  | 1/2 (50%)     |
| MIC                       | 0.8870   | −0.0006     | N/A      | 0/3 (0%)      |
| splice                    | 0.9614   | N/A         | N/A      | N/A           |
| hiva_agnostic             | 0.9649   | +0.0000     | +0.0000  | 0/5 (0%)      |
| E-CommerceShippingData    | 0.6759   | +0.0012     | +0.0032  | 1/2 (50%)     |
| online_shoppers_intention | 0.9033   | +0.0005     | −0.0013  | 2/3 (67%)     |
| Amazon_employee_access    | 0.9431   | −0.0003     | −0.0004  | 0/2 (0%)      |
| APSFailure                | 0.9914   | −0.0008     | −0.0008  | 0/4 (0%)      |

**Overall Direction Accuracy: 4/21 (19.0%)**

## Analysis

### Positive Findings
- **maternal_health_risk**: The combined pipeline (YeoJohnson → ArithmeticInteractions)
  achieved +0.0138 gain, the best result across all datasets. This demonstrates that
  method combination can produce synergistic effects even when individual methods underperform.
- **E-CommerceShippingData**: Both individual (ArithmeticInteractions: +0.0012) and
  combined (+0.0032) pipelines showed genuine improvement on the lowest-baseline dataset.

### Identified Limitations
1. **Over-optimistic on high-baseline datasets**: When baseline accuracy exceeds ~0.90,
   the meta-model still predicts positive gains, but preprocessing rarely helps
   (APSFailure 0/4, hiva_agnostic 0/5, Amazon_employee_access 0/2).
2. **Poor out-of-distribution generalization**: Training Win Accuracy (40–82%) did not
   transfer to test datasets (19% overall), suggesting the meta-model overfits to
   CC18-specific patterns with only 66 training datasets.
3. **Limited method coverage**: splice (all-categorical, 60 columns) had zero applicable
   recommended methods, exposing gaps in the 10-method repertoire.
4. **Combined pipeline risk**: When a harmful method enters the pipeline, it degrades
   overall performance (online_shoppers: −0.0013, Amazon: −0.0004).

## Conclusion

The meta-learning framework successfully identifies beneficial preprocessing for
low-to-moderate baseline datasets (maternal_health_risk, E-CommerceShippingData),
but struggles with high-baseline or structurally different datasets. The 19% direction
accuracy on out-of-distribution data highlights the need for more diverse training
datasets, richer meta-features, and potentially a confidence threshold to abstain
from recommendations when the meta-model is uncertain.